In [ ]:
import pandas as pd

In [ ]:
cars = pd.read_csv("./quikr_car.csv")

In [ ]:
cars.head()

In [ ]:
cars["company"].unique()

In [ ]:
cars["year"].unique()

In [ ]:
cars["year"] = cars["year"].astype(str)

In [ ]:
cars["year"].unique()

In [ ]:
cars = cars[cars["year"].str.isnumeric()]

In [ ]:
cars.head()

In [ ]:
cars["year"].unique()

In [ ]:
cars["year"] = cars["year"].astype(int)

In [ ]:
cars.info()

In [ ]:
cars = cars[cars["Price"] != "Ask For Price"]

In [ ]:
cars["Price"] = cars["Price"].str.replace(",", "").astype(int)

In [ ]:
cars.info()

In [ ]:
cars["kms_driven"] = cars["kms_driven"].str.split().str.get(0).str.replace(",", "")

In [ ]:
cars["kms_driven"] = cars["kms_driven"].astype(str)

In [ ]:
cars = cars[cars["kms_driven"].str.isnumeric()]

In [ ]:
cars["kms_driven"] = cars["kms_driven"].astype(int)

In [ ]:
cars.info()

In [ ]:
cars["fuel_type"].unique()

In [ ]:
cars = cars[~cars["fuel_type"].isna()]

In [ ]:
cars.head()

In [ ]:
cars.info()

In [ ]:
cars["name"] = cars["name"].str.split().str.slice(start=0, stop=3).str.join(" ")

In [ ]:
cars = cars.reset_index(drop=True)

In [ ]:
cars

In [ ]:
cars.to_csv("cleaned_car.csv")

In [ ]:
cars = cars[cars["Price"]<6000000]

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score

In [ ]:
X = cars[["name", "company", "fuel_type", "year"]]
Y = cars["Price"]

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(X,Y,test_size=0.2)

In [ ]:
x_train

In [ ]:
ohe = OneHotEncoder()
ohe.fit(X[["name", "company", "fuel_type"]])

In [ ]:
column_transformed = make_column_transformer((OneHotEncoder(categories=ohe.categories_), ["name", "company", "fuel_type"]),
                                            remainder="passthrough")

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
lr = LinearRegression()

In [ ]:
pipe = make_pipeline(column_transformed, lr)

In [ ]:
pipe.fit(x_train, y_train)

In [ ]:
y_pred = pipe.predict(x_test)

In [ ]:
score = r2_score(y_pred, y_test)

In [ ]:
score

In [ ]:
scores=[]
for i in range(1000):
    X_train,X_test,y_train,y_test=train_test_split(X,Y,test_size=0.1,random_state=i)
    lr=LinearRegression()
    pipe=make_pipeline(column_transformed,lr)
    pipe.fit(X_train,y_train)
    y_pred=pipe.predict(X_test)
    scores.append(r2_score(y_test,y_pred))

In [ ]:
import numpy as np

In [ ]:
np.argmax(scores)

In [ ]:
scores[321]

# Best model with highest accuracy

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,Y,test_size=0.1,random_state=np.argmax(scores))
lr=LinearRegression()
pipe=make_pipeline(column_transformed,lr)
pipe.fit(X_train,y_train)
y_pred=pipe.predict(X_test)
r2_score(y_test,y_pred)

In [ ]:
import pickle
pickle.dump(pipe, open("car_predictor.pkl", "wb"))